In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import joblib
import os
from datetime import datetime, timedelta
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# Define the artifacts directory globally or as a common variable
ARTIFACTS_DIR = "Scenario 3"

def prepare_data(df):
    """
    Performs initial cleaning and feature engineering on a loaded DataFrame.
    Defines 'purchased' flag based on Quantity > 0.
    Optimized for memory and speed.

    Args:
        df (pandas.DataFrame): The raw DataFrame containing the sales data.

    Returns:
        tuple: A tuple containing:
            - pandas.DataFrame: The prepared DataFrame with 'purchased' flag.
            - pandas.DataFrame: A DataFrame of all unique products.
            - pandas.DataFrame: A DataFrame of all unique customer attributes.
    """
    print(f"Starting data preparation on loaded DataFrame. Initial shape: {df.shape}")
    initial_memory_usage = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Initial DataFrame memory usage: {initial_memory_usage:.2f} MB")

    # Ensure essential columns are present before proceeding
    required_cols = ['Order Date', 'Customer ID', 'Product ID', 'Quantity']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in the input DataFrame.")

    # Convert date columns to datetime objects (vectorized)
    for col in ['Order Date', 'Ship Date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

    # Drop rows where essential data is NaT or NaN (vectorized)
    initial_rows = df.shape[0]
    df.dropna(subset=required_cols, inplace=True)
    if df.shape[0] < initial_rows:
        print(f"Dropped {initial_rows - df.shape[0]} rows due to missing essential data ({', '.join(required_cols)}).")

    # Downcast numeric columns (vectorized where possible)
    for col in ['Sales', 'Discount', 'Profit', 'Row ID', 'Postal Code']: # Quantity is handled separately
        if col in df.columns:
            if df[col].dtype == 'int64':
                df[col] = pd.to_numeric(df[col], downcast='integer')
            elif df[col].dtype == 'float64':
                df[col] = pd.to_numeric(df[col], downcast='float')

    # Convert Quantity to appropriate smaller integer type and define 'purchased' flag
    if 'Quantity' in df.columns:
        df['Quantity'] = df['Quantity'].astype('int16')
        df['purchased'] = (df['Quantity'] > 0).astype('int8') # Use int8 for boolean flag
        print(f"Created 'purchased' flag based on Quantity > 0. Number of purchases: {df['purchased'].sum()}")

    # --- Feature Engineering (vectorized) ---
    if 'Order Date' in df.columns:
        df['Order_Year'] = df['Order Date'].dt.year.astype('int16')
        df['Order_Month'] = df['Order Date'].dt.month.astype('int8')
        df['Order_Day'] = df['Order Date'].dt.day.astype('int8')

    if 'Order Date' in df.columns and 'Ship Date' in df.columns:
        # Using .dt.days is vectorized. .fillna(0) then .astype('int16') is efficient.
        df['Days_to_Ship'] = (df['Ship Date'] - df['Order Date']).dt.days.fillna(0).astype('int16')

    # Identify unique products and customers for later use
    # Selecting only necessary columns before drop_duplicates can be slightly faster for wide DFs
    unique_products_df = df[['Product Name', 'Category', 'Sub-Category', 'Product ID']].drop_duplicates().reset_index(drop=True)
    unique_customers_df = df[['Customer ID', 'Segment', 'Country', 'City', 'State', 'Region']].drop_duplicates().reset_index(drop=True)

    # The 'full_dataset' for training is now the cleaned and feature-engineered 'df' itself
    columns_to_keep_for_model = [
        'Customer ID', 'Product ID', 'Product Name', 'Category', 'Sub-Category',
        'Segment', 'Country', 'City', 'State', 'Region', 'purchased',
        'Order_Year', 'Order_Month', 'Order_Day', 'Days_to_Ship'
    ]
    # Filter for columns that actually exist in the DataFrame
    columns_to_keep_for_model = [col for col in columns_to_keep_for_model if col in df.columns]
    full_dataset = df[columns_to_keep_for_model].copy() # Ensure a copy to avoid SettingWithCopyWarning

    final_memory_usage = full_dataset.memory_usage(deep=True).sum() / (1024**2)
    print(f"Final prepared dataset memory usage: {final_memory_usage:.2f} MB")
    print(f"Data preparation complete. Final interaction dataset shape: {full_dataset.shape}")
    return full_dataset, unique_products_df, unique_customers_df


### 2. `encode`
def encode(df, unique_products_df, unique_customers_df):
    """
    Encodes categorical features using Label Encoding.
    Optimized for faster execution by batch processing and vectorized operations.

    Args:
        df (pandas.DataFrame): The prepared DataFrame.
        unique_products_df (pandas.DataFrame): DataFrame of unique products.
        unique_customers_df (pandas.DataFrame): DataFrame of unique customers.

    Returns:
        tuple: A tuple containing:
            - pandas.DataFrame: The encoded DataFrame.
            - dict: A dictionary of fitted LabelEncoders.
            - list: List of features used for training.
            - pandas.DataFrame: Encoded unique products DataFrame for prediction.
            - pandas.DataFrame: Encoded unique customers DataFrame for prediction.
    """
    print("Starting feature encoding with LabelEncoder (optimized version)...")
    initial_memory_usage = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memory usage before encoding: {initial_memory_usage:.2f} MB")

    encoded_df = df.copy()
    label_encoders = {}
    features = []

    # Define all categorical columns that need to be encoded
    categorical_cols = [
        'Customer ID', 'Product ID', 'Product Name', 'Category', 'Sub-Category',
        'Segment', 'Country', 'City', 'State', 'Region'
    ]

    for col in categorical_cols:
        if col in encoded_df.columns:
            le = LabelEncoder()
            # Combine all unique values from relevant dataframes for fitting the encoder
            # This ensures the encoder learns all possible categories across all datasets.
            # Using .astype(str) to avoid issues with mixed types or non-string objects.
            all_values = pd.concat([
                encoded_df[col].astype(str),
                unique_products_df[col].astype(str) if col in unique_products_df.columns else pd.Series(),
                unique_customers_df[col].astype(str) if col in unique_customers_df.columns else pd.Series()
            ]).unique()

            le.fit(all_values)
            label_encoders[col] = le

            # Apply transform to the main DataFrame
            encoded_df[f'{col}_encoded'] = le.transform(encoded_df[col].astype(str))
            features.append(f'{col}_encoded')
            print(f"Label encoded '{col}'.")
        else:
            print(f"Warning: Categorical column '{col}' not found in the main DataFrame for encoding.")


    # Add numerical features to the list of features
    numerical_features = ['Order_Year', 'Order_Month', 'Order_Day', 'Days_to_Ship']
    for num_col in numerical_features:
        if num_col in encoded_df.columns:
            features.append(num_col)

    # Now, encode the unique_products_df and unique_customers_df using the *fitted* encoders
    encoded_unique_products_df = unique_products_df.copy()
    encoded_unique_customers_df = unique_customers_df.copy()

    # Apply encoding to unique products dataframe
    product_cols_to_encode = ['Product Name', 'Category', 'Sub-Category', 'Product ID']
    for col in product_cols_to_encode:
        if col in unique_products_df.columns and col in label_encoders:
            le = label_encoders[col]
            # Use a more efficient way to handle potential unseen labels if they existed,
            # but given how unique_products_df is generated, they should be covered.
            # If not, a mapping with .get() or .mask() is better than .apply(lambda).
            # For simplicity and speed, assuming all labels are known based on combined fitting.
            encoded_unique_products_df[f'{col}_encoded'] = le.transform(unique_products_df[col].astype(str))
            # No need to drop original columns here, they might be useful for identification

    # Apply encoding to unique customers dataframe
    customer_cols_to_encode = ['Customer ID', 'Segment', 'Country', 'City', 'State', 'Region']
    for col in customer_cols_to_encode:
        if col in unique_customers_df.columns and col in label_encoders:
            le = label_encoders[col]
            encoded_unique_customers_df[f'{col}_encoded'] = le.transform(unique_customers_df[col].astype(str))
            # No need to drop original columns here

    final_memory_usage = encoded_df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memory usage after encoding: {final_memory_usage:.2f} MB")
    print("Feature encoding complete.")
    return encoded_df, label_encoders, features, encoded_unique_products_df, encoded_unique_customers_df


### 3. `build_model` (Already efficient)
def build_model(input_features_count):
    """
    Builds the machine learning model. For this example, we'll use Logistic Regression.

    Args:
        input_features_count (int): The number of input features for the model.

    Returns:
        sklearn.linear_model.LogisticRegression: An unfitted Logistic Regression model.
    """
    print(f"Building Logistic Regression model with {input_features_count} input features...")
    # Consider using 'saga' solver for larger datasets if 'liblinear' is too slow,
    # but it might require feature scaling. For now, liblinear is generally robust.
    model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000, n_jobs=-1)
    print("Model built successfully.")
    return model


### 4. `train_model` (Already efficient)
def train_model(model, encoded_df, features):
    """
    Trains the provided machine learning model.

    Args:
        model: The unfitted machine learning model.
        encoded_df (pandas.DataFrame): The DataFrame with encoded features and target.
        features (list): A list of column names to be used as features for training.

    Returns:
        tuple: A tuple containing:
            - trained_model: The fitted model.
            - pandas.DataFrame: The X_test dataset for evaluation.
            - pandas.Series: The y_test series for evaluation.
    """
    print("Starting model training...")

    # Define features (X) and target (y)
    X = encoded_df[features]
    y = encoded_df['purchased']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Data split: X_train shape {X_train.shape}, X_test shape {X_test.shape}")

    # Train the model
    model.fit(X_train, y_train)
    print("Model training complete.")

    return model, X_test, y_test


### 5. `evaluate_model` (Already efficient)
def evaluate_model(model, X_test, y_test):
    """
    Evaluates the performance of the trained model.

    Args:
        model: The trained machine learning model.
        X_test (pandas.DataFrame): The test features.
        y_test (pandas.Series): The true labels for the test set.

    Returns:
        dict: A dictionary containing evaluation metrics.
    """
    print("Evaluating model performance...")

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Probability of the positive class

    metrics = {
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_prob),
        'Average Precision': average_precision_score(y_test, y_prob)
    }

    print("Model evaluation complete. Metrics:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

    return metrics


### 6. `save_artifacts`
def save_artifacts(model, label_encoders, features, all_products_for_prediction_df):
    """
    Saves the trained model and encoding objects, and unique products DataFrame to disk.

    Args:
        model: The trained machine learning model.
        label_encoders (dict): Dictionary of fitted LabelEncoders.
        features (list): List of features used for training.
        all_products_for_prediction_df (pd.DataFrame): DataFrame of all unique products.
    """
    print(f"Saving model artifacts to '{ARTIFACTS_DIR}'...")
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)
    joblib.dump(model, os.path.join(ARTIFACTS_DIR, 'recommendation_model.pkl'))
    joblib.dump(label_encoders, os.path.join(ARTIFACTS_DIR, 'label_encoders.pkl'))
    joblib.dump(features, os.path.join(ARTIFACTS_DIR, 'model_features.pkl'))
    joblib.dump(all_products_for_prediction_df, os.path.join(ARTIFACTS_DIR, 'all_products_df.pkl')) # New line
    print("Model artifacts saved successfully.")


### 7. `load_artifacts`
def load_artifacts():
    """
    Loads the trained model, encoding objects, and unique products DataFrame from disk.

    Returns:
        tuple: A tuple containing:
            - model: The loaded trained machine learning model.
            - label_encoders (dict): Loaded LabelEncoders.
            - features (list): List of features used for training.
            - all_products_df (pd.DataFrame): Loaded DataFrame of all unique products.
    """
    print(f"Loading model artifacts from '{ARTIFACTS_DIR}'...")
    try:
        model = joblib.load(os.path.join(ARTIFACTS_DIR, 'recommendation_model.pkl'))
        label_encoders = joblib.load(os.path.join(ARTIFACTS_DIR, 'label_encoders.pkl'))
        features = joblib.load(os.path.join(ARTIFACTS_DIR, 'model_features.pkl'))
        all_products_df = joblib.load(os.path.join(ARTIFACTS_DIR, 'all_products_df.pkl')) # New line
        print("Model artifacts loaded successfully.")
        return model, label_encoders, features, all_products_df
    except FileNotFoundError:
        print(f"Error: Could not find artifacts in '{ARTIFACTS_DIR}'. Please ensure they are saved.")
        return None, None, None, None # Return None for the new artifact too
    except Exception as e:
        print(f"An error occurred while loading artifacts: {e}")
        return None, None, None, None


### 8. `predictions`
def predictions(customer_data, forecast_period): # all_products_df removed from args
    """
    Predicts the top 5 products a given customer may buy for the next few forecast months
    with their probability of purchase.
    Loads model artifacts internally, including the all_products_df.

    Args:
        customer_data (dict): Dictionary of customer attributes (e.g., {"Customer ID": "Customer B", "Segment": "Corporate"}).
        forecast_period (int): The number of months to forecast.

    Returns:
        dict: Dictionary with forecast months as keys and values as dictionaries
              mapping product names to purchase probabilities (top 5).
    """
    # --- PERFORMANCE NOTE ---
    # Loading artifacts inside this function will cause disk I/O every time it's called.
    # For high-frequency prediction services, it's more efficient to load artifacts once
    # outside this function and keep them in memory.
    # For demonstration or infrequent calls, this approach is simpler.
    print("Attempting to load model artifacts for prediction...")
    model, label_encoders, model_features, all_products_df = load_artifacts() # all_products_df added here

    if model is None or label_encoders is None or model_features is None or all_products_df is None:
        print("Failed to load necessary artifacts for prediction. Cannot proceed.")
        return {}

    print(f"Generating predictions for Customer ID: {customer_data.get('Customer ID')} for {forecast_period} months...")

    customer_id = customer_data.get("Customer ID")
    customer_segment = customer_data.get("Segment")
    customer_country = customer_data.get("Country", "Unknown")
    customer_city = customer_data.get("City", "Unknown")
    customer_state = customer_data.get("State", "Unknown")
    customer_region = customer_data.get("Region", "Unknown")

    if not customer_id:
        print("Error: 'Customer ID' not provided in customer_data.")
        return {}

    temp_df_data = {
        'Customer ID': [customer_id] * len(all_products_df),
        'Product ID': all_products_df['Product ID'],
        'Product Name': all_products_df['Product Name'],
        'Category': all_products_df['Category'],
        'Sub-Category': all_products_df['Sub-Category'],
        'Segment': [customer_segment] * len(all_products_df),
        'Country': [customer_country] * len(all_products_df),
        'City': [customer_city] * len(all_products_df),
        'State': [customer_state] * len(all_products_df),
        'Region': [customer_region] * len(all_products_df)
    }
    inference_df = pd.DataFrame(temp_df_data)

    categorical_cols_to_encode = [
        'Customer ID', 'Product ID', 'Product Name', 'Category', 'Sub-Category',
        'Segment', 'Country', 'City', 'State', 'Region'
    ]

    for col in categorical_cols_to_encode:
        encoded_col_name = f'{col}_encoded'
        if col in label_encoders and col in inference_df.columns:
            le = label_encoders[col]
            mapping_dict = {label: encoded_val for encoded_val, label in enumerate(le.classes_)}
            inference_df[encoded_col_name] = inference_df[col].astype(str).map(mapping_dict).fillna(-1).astype(int)
        else:
            inference_df[encoded_col_name] = -1
            print(f"Warning: LabelEncoder for '{col}' not found or column not in inference data. Using -1 for encoding.")

    current_time = datetime.now() # Current time is 2025-07-25 15:10:06 IST.
    inference_df['Order_Year'] = current_time.year
    inference_df['Order_Month'] = current_time.month
    inference_df['Order_Day'] = current_time.day
    inference_df['Days_to_Ship'] = 3

    X_predict_cols = [f for f in model_features if f in inference_df.columns]
    X_predict = inference_df[X_predict_cols]

    probabilities = model.predict_proba(X_predict)[:, 1]
    inference_df['probability'] = probabilities

    top_products_all = inference_df.nlargest(5, 'probability')

    forecast_results = {}
    for i in range(forecast_period):
        forecast_month_date = current_time.replace(day=1) + timedelta(days=30 * (i + 1))
        forecast_month = forecast_month_date.strftime("%Y-%m")
        month_predictions = {}
        for _, row in top_products_all.iterrows():
            month_predictions[row['Product Name']] = round(row['probability'], 2)
        forecast_results[forecast_month] = month_predictions

    print("Predictions generated.")
    return forecast_results



In [23]:

# --- Main Execution Block ---
if __name__ == "__main__":
    # --- 0. Create a Dummy DataFrame for Demonstration ---
    # Instead of reading from CSV, we directly create the DataFrame
    # raw_df = pd.read_csv()
    raw_df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Complete.csv')
    print("Loaded DataFrame.")

    # --- 1. Prepare Data ---
    prepared_df, all_products_for_prediction_df, all_customers_df = prepare_data(raw_df)

    # --- 2. Encode Data ---
    encoded_df, label_encoders, model_features, \
    encoded_unique_products_df, encoded_unique_customers_df = encode(
        prepared_df, all_products_for_prediction_df, all_customers_df
    )

    print(f"Final features for model training: {model_features}")

    # --- 3. Build Model ---
    model = build_model(len(model_features))

    # --- 4. Train Model ---
    trained_model, X_test, y_test = train_model(model, encoded_df, model_features)

    # --- 5. Evaluate Model ---
    evaluation_metrics = evaluate_model(trained_model, X_test, y_test)

    # --- 6. Save Artifacts ---
    # Pass all_products_for_prediction_df to save_artifacts
    save_artifacts(trained_model, label_encoders, model_features, all_products_for_prediction_df)

    # --- 7. Make Predictions (Calling the new `predictions` function) ---
    customer_input = {
        "Customer ID": "Customer B",
        "Segment": "Corporate",
        "Country": "United States",
        "City": "Los Angeles",
        "State": "California",
        "Region": "West",
    }
    forecast_months = 2

    # Now, all_products_for_prediction_df is NOT passed to predictions
    predictions_output = predictions(
        customer_input,
        forecast_months
    )
    print("\nCustomer Recommendation Predictions:")
    print(predictions_output)
    
    
    # # --- 6. Save Artifacts ---
    # save_artifacts(trained_model, label_encoders, model_features)

    # # --- 7. Load Artifacts (Demonstration) ---
    # loaded_model, loaded_label_encoders, loaded_features, all_products_for_prediction_df = load_artifacts()
    
    # # --- 8. Make Predictions ---
    # customer_input = {
    #     "Customer ID": "Customer B",
    #     "Segment": "Corporate",
    #     "Country": "United States",
    #     "City": "Los Angeles",
    #     "State": "California",
    #     "Region": "West",
    # }
    # forecast_months = 5

    # predictions_output = predictions(
    #     customer_input,
    #     all_products_for_prediction_df,
    #     forecast_months,
    #     loaded_model,
    #     loaded_label_encoders,
    #     loaded_features
    # )
    # print("\nCustomer Recommendation Predictions:")
    # print(predictions_output)


Loaded DataFrame.
Starting data preparation on loaded DataFrame. Initial shape: (49970, 22)
Initial DataFrame memory usage: 45.36 MB
Created 'purchased' flag based on Quantity > 0. Number of purchases: 33980
Final prepared dataset memory usage: 28.61 MB
Data preparation complete. Final interaction dataset shape: (49970, 15)
Starting feature encoding with LabelEncoder (optimized version)...
Memory usage before encoding: 28.61 MB
Label encoded 'Customer ID'.
Label encoded 'Product ID'.
Label encoded 'Product Name'.
Label encoded 'Category'.
Label encoded 'Sub-Category'.
Label encoded 'Segment'.
Label encoded 'Country'.
Label encoded 'City'.
Label encoded 'State'.
Label encoded 'Region'.
Memory usage after encoding: 32.42 MB
Feature encoding complete.
Final features for model training: ['Customer ID_encoded', 'Product ID_encoded', 'Product Name_encoded', 'Category_encoded', 'Sub-Category_encoded', 'Segment_encoded', 'Country_encoded', 'City_encoded', 'State_encoded', 'Region_encoded', 'Or

In [21]:
predictions_output.keys()


dict_keys(['2025-07', '2025-08', '2025-09', '2025-10', '2025-11'])